In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

In [ ]:
# Simple Transformer Classifier using PyTorch's Built-in Modules
# This implementation uses `nn.MultiheadAttention` and `nn.TransformerEncoder` 
# so you don't need to implement attention details yourself.

In [ ]:
class SimpleTransformerClassifier(nn.Module):
    """
    Simple transformer for classification tasks using PyTorch's built-in modules.
    No need to implement self-attention or cross-attention details!
    """
    def __init__(self, d_model=512, nhead=8, num_layers=6, dim_feedforward=2048, 
                 num_classes=2, dropout=0.1, max_seq_len=512):
        super().__init__()
        
        # Use PyTorch's built-in TransformerEncoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation='gelu',
            batch_first=True  # (batch, seq, features) instead of (seq, batch, features)
        )
        
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        # Classification head
        self.classifier = nn.Linear(d_model, num_classes)
        
        # Optional: learnable positional encoding
        self.pos_encoding = nn.Parameter(torch.randn(1, max_seq_len, d_model))
        
    def forward(self, x, mask=None):
        """
        Args:
            x: (batch_size, seq_len, d_model) - input embeddings
            mask: (batch_size, seq_len) - attention mask (True for padding tokens)
        Returns:
            logits: (batch_size, num_classes)
        """
        batch_size, seq_len, _ = x.shape
        
        # Add positional encoding
        x = x + self.pos_encoding[:, :seq_len, :]
        
        # Create attention mask if provided
        # PyTorch expects mask where True means "ignore this position"
        attn_mask = None
        if mask is not None:
            # Convert to format expected by TransformerEncoder
            # True positions will be masked out
            attn_mask = mask.bool()
        
        # Pass through transformer
        # TransformerEncoder handles all the attention details internally!
        x = self.transformer(x, src_key_padding_mask=attn_mask)
        
        # Global average pooling over sequence dimension
        # You could also use the [CLS] token or max pooling
        x = x.mean(dim=1)  # (batch_size, d_model)
        
        # Classification
        logits = self.classifier(x)  # (batch_size, num_classes)
        
        return logits

In [ ]:
# Example usage
batch_size = 32
seq_len = 128
d_model = 512
num_classes = 2

# Create model
model = SimpleTransformerClassifier(
    d_model=d_model,
    nhead=8,
    num_layers=6,
    dim_feedforward=2048,
    num_classes=num_classes,
    dropout=0.1
)

# Dummy input (pretend these are embeddings from a tokenizer)
x = torch.randn(batch_size, seq_len, d_model)

# Forward pass
with torch.no_grad():
    logits = model(x)
    
print(f"Input shape: {x.shape}")
print(f"Output logits shape: {logits.shape}")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

## Alternative: Even Simpler Version with Just MultiheadAttention

In [ ]:
class MinimalTransformerClassifier(nn.Module):
    """
    Minimal transformer using just nn.MultiheadAttention.
    Even simpler - just one attention layer!
    """
    def __init__(self, d_model=512, nhead=8, num_classes=2, dropout=0.1):
        super().__init__()
        
        # Use PyTorch's built-in MultiheadAttention
        self.attention = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=nhead,
            dropout=dropout,
            batch_first=True  # (batch, seq, features)
        )
        
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        
        # Feed-forward network
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 4, d_model),
            nn.Dropout(dropout)
        )
        
        # Classification head
        self.classifier = nn.Linear(d_model, num_classes)
        
    def forward(self, x, mask=None):
        """
        Args:
            x: (batch_size, seq_len, d_model)
            mask: (batch_size, seq_len) - True for padding tokens
        """
        # Self-attention with residual connection
        residual = x
        x = self.norm1(x)
        x, _ = self.attention(x, x, x, key_padding_mask=mask)
        x = x + residual
        
        # Feed-forward with residual connection
        residual = x
        x = self.norm2(x)
        x = self.ffn(x)
        x = x + residual
        
        # Pool and classify
        x = x.mean(dim=1)  # Global average pooling
        logits = self.classifier(x)
        
        return logits

In [ ]:
# Test the minimal version
minimal_model = MinimalTransformerClassifier(d_model=512, nhead=8, num_classes=2)

x = torch.randn(32, 128, 512)
with torch.no_grad():
    logits = minimal_model(x)
    
print(f"Minimal model output shape: {logits.shape}")
print(f"Minimal model parameters: {sum(p.numel() for p in minimal_model.parameters()):,}")

## Training Example

In [ ]:
# Training setup
num_samples = 1000
batch_size = 32
learning_rate = 1e-4
num_epochs = 10
d_model = 256
seq_len = 64
num_classes = 2

# Create model
model = SimpleTransformerClassifier(
    d_model=d_model,
    nhead=8,
    num_layers=4,
    dim_feedforward=1024,
    num_classes=num_classes,
    max_seq_len=seq_len
)

# Dummy data (in practice, these would be real embeddings)
x_train = torch.randn(num_samples, seq_len, d_model)
y_train = torch.randint(0, num_classes, (num_samples,))

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

# DataLoader
dataset = torch.utils.data.TensorDataset(x_train, y_train)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)

print("Starting training...")
print(f"Model has {sum(p.numel() for p in model.parameters()):,} parameters")

In [ ]:
# Training loop
for epoch in range(num_epochs):
    running_loss = 0.0
    for inputs, labels in dataloader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    
    avg_loss = running_loss / len(dataloader)
    if (epoch + 1) % 2 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}')

print('Training finished!')

## Key Advantages

1. **No attention implementation needed**: Uses `nn.MultiheadAttention` or `nn.TransformerEncoder`
2. **Battle-tested**: PyTorch's implementations are optimized and well-tested
3. **Flexible**: Easy to adjust `nhead`, `num_layers`, `d_model`, etc.
4. **Supports masking**: Can handle padding masks automatically
5. **Simple API**: Just pass embeddings and get classification logits